1. cargar directorio

In [21]:
import pandas as pd

directorio_mineduc = pd.read_csv('../raw/20250926_Directorio_Oficial_EE_2025_20250430_WEB.csv', sep=';')


In [22]:
directorio_mineduc.head()

,AGNO,RBD,DGV_RBD,NOM_RBD,MRUN,RUT_SOSTENEDOR,P_JURIDICA,COD_REG_RBD,NOM_REG_RBD_A,COD_PRO_RBD,...,ESPE_02,ESPE_03,ESPE_04,ESPE_05,ESPE_06,ESPE_07,ESPE_08,ESPE_09,ESPE_10,ESPE_11
0,2025,1,9,LICEO POLITECNICO ARICA,,62000660,1,15,AYP,151,...,52009,52010,52013,53014,53015,61002,61003,62004,64001,0
1,2025,2,7,PARVULARIO LAS ESPIGUITAS,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0
2,2025,3,5,ESC. PEDRO VICENTE GUTIERREZ TORRES,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0
3,2025,4,3,LICEO OCTAVIO PALMA PEREZ,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0
4,2025,5,1,JOVINA NARANJO FERNANDEZ,,62000660,1,15,AYP,151,...,0,0,0,0,0,0,0,0,0,0


2.Filtrar por la comuna de Pudahuel (MVP)

In [23]:
directorio_mineduc[directorio_mineduc['NOM_COM_RBD'].str.contains('PUDAHUEL', case=False, na=False)]['NOM_COM_RBD'].unique()

<StringArray>
['PUDAHUEL']
Length: 1, dtype: str

In [24]:
directorio_mineduc[directorio_mineduc['NOM_COM_RBD']=='PUDAHUEL']['COD_COM_RBD'].unique()

array([13124])

In [25]:
directorio_mineduc_pudahuel = directorio_mineduc[directorio_mineduc['COD_COM_RBD']==13124]
len(directorio_mineduc_pudahuel)

85

3.Verificar valores de ESTADO_ESTAB

In [26]:
directorio_mineduc_pudahuel['ESTADO_ESTAB'].value_counts()

ESTADO_ESTAB
1    68
3    16
2     1
Name: count, dtype: int64

In [27]:
directorio_mineduc_activos = directorio_mineduc_pudahuel[
(directorio_mineduc_pudahuel['ESTADO_ESTAB'] == 1) & (directorio_mineduc_pudahuel['MAT_TOTAL'] > 0)
]
len(directorio_mineduc_activos)

62

In [28]:
CODIGOS_REGULAR = [110, 310, 410, 510, 610, 710, 810, 910]
columnas_ens = ['ENS_01', 'ENS_02', 'ENS_03', 'ENS_04', 'ENS_05', 'ENS_06', 'ENS_07', 'ENS_08', 'ENS_09', 'ENS_10', 'ENS_11']

directorio_mineduc_activos['ofrece_educacion_regular'] = directorio_mineduc_activos[columnas_ens].isin(CODIGOS_REGULAR).any(axis=1)

In [29]:
directorio_mineduc_activos['ofrece_educacion_regular'].value_counts()

ofrece_educacion_regular
True     45
False    17
Name: count, dtype: int64

### Nivel: ¿Básica, Media o ambos?

Mismo patrón que `ofrece_educacion_regular` (celda anterior), separando el código 110 (Básica) del resto de códigos de Media (310, 410, 510, 610, 710, 810, 910).

In [30]:
CODIGOS_BASICA = [110]
CODIGOS_MEDIA = [310, 410, 510, 610, 710, 810, 910]

directorio_mineduc_activos['ofrece_basica'] = directorio_mineduc_activos[columnas_ens].isin(CODIGOS_BASICA).any(axis=1)
directorio_mineduc_activos['ofrece_media'] = directorio_mineduc_activos[columnas_ens].isin(CODIGOS_MEDIA).any(axis=1)

In [31]:
directorio_mineduc_activos[['ofrece_basica','ofrece_media']].value_counts()

ofrece_basica  ofrece_media
True           False           30
False          False           17
True           True            13
False          True             2
Name: count, dtype: int64

### Modalidad: Educación Especial

Códigos 211-219 (por tipo de discapacidad) y 299 (Programa de Integración Escolar).

In [32]:
CODIGOS_ESPECIAL = [211, 212, 213, 214, 215, 216, 217, 218, 219, 299]

directorio_mineduc_activos['ofrece_educacion_especial'] = directorio_mineduc_activos[columnas_ens].isin(CODIGOS_ESPECIAL).any(axis=1)

In [33]:
directorio_mineduc_activos['ofrece_educacion_especial'].value_counts()

ofrece_educacion_especial
False    46
True     16
Name: count, dtype: int64

### Definir el universo final

Un colegio entra al universo si ofrece educación regular **o** especial, excluyendo el caso borde RBD 25564 (revisado en sesión anterior).

In [34]:
directorio_mineduc_universo = directorio_mineduc_activos[
(directorio_mineduc_activos['ofrece_educacion_regular'] | directorio_mineduc_activos['ofrece_educacion_especial']) & (directorio_mineduc_activos['RBD'] !=25564)
] 
len(directorio_mineduc_universo)                                                    

57

4. unificar directorio_mineduc_universo con colegios_enriquecidos

In [35]:
colegios_enriquecidos = pd.read_json('../../web-app/public/data/colegios_enriquecidos.json')

columnas_nuevas = directorio_mineduc_universo [['RBD', 'ofrece_educacion_regular', 'ofrece_educacion_especial', 'ofrece_basica', 'ofrece_media', 'CONVENIO_PIE']]

colegios_universo = pd.merge(colegios_enriquecidos, columnas_nuevas, left_on='rbd', right_on='RBD', how='inner')

len(colegios_universo)

57

In [36]:
colegios_universo = colegios_universo.drop(columns=['RBD'])

In [37]:
orden_pago = {
    'GRATUITO': 0,
    '$1.000 A $10.000': 1,
    '$10.001 A $25.000': 2,
    '$25.001 A $50.000': 3,
    '$50.001 A $100.000': 4,
    'MAS DE $100.000': 5
}

colegios_universo['pago_mensual_rango'] = colegios_universo['PAGO_MENSUAL'].map(orden_pago)
colegios_universo['pago_mensual_rango'].isna().sum()

np.int64(2)

In [38]:
colegios_universo.head()

,rbd,NOM_RBD,ESTADO_ESTAB,NOM_COM_RBD,ORI_RELIGIOSA,ORI_OTRO_GLOSA,PAGO_MATRICULA,PAGO_MENSUAL,COD_DEPE,LATITUD,...,autoestima_prom_8b,clima_prom_8b,participacion_prom_8b,habitos_prom_8b,ofrece_educacion_regular,ofrece_educacion_especial,ofrece_basica,ofrece_media,CONVENIO_PIE,pago_mensual_rango
0,10077,LICEO CENTRO EXPERIMENTAL PUDAHUEL CAREN,1,PUDAHUEL,7,LAICA,GRATUITO,GRATUITO,6,-33.444558,...,NaN,NaN,NaN,NaN,True,False,False,True,1,0.0
1,10080,ESCUELA TENIENTE HERNAN MERINO CORREA,1,PUDAHUEL,1,,GRATUITO,GRATUITO,6,-33.432563,...,82.0,83.0,87.0,78.0,True,False,True,False,1,0.0
2,10081,ESCUELA ESTRELLA DE CHILE,1,PUDAHUEL,1,,GRATUITO,GRATUITO,6,-33.436599,...,75.0,71.0,74.0,72.0,True,False,True,False,1,0.0
3,10085,ESCUELA ESTADO DE FLORIDA,1,PUDAHUEL,7,LAICA,GRATUITO,GRATUITO,6,-33.432981,...,66.0,71.0,67.0,62.0,True,False,True,False,1,0.0
4,10090,ESCUELA ALEXANDER GRAHAM BELL,1,PUDAHUEL,1,,GRATUITO,GRATUITO,6,-33.432629,...,70.0,70.0,74.0,67.0,True,False,True,False,1,0.0


In [39]:
colegios_universo.shape

(57, 88)

In [40]:
colegios_universo.to_json('../../web-app/public/data/colegios_universo.json', orient='records', indent=2, force_ascii=False)